# Introduction

In this notebook, we intend to teach Retrieval-Augmented Generation(RAG) concepts from scratch with coding and finally a project.

# Definition

One of the problems with large language models is outdated information.

That is, these language models do not have up-to-date information.

---

## Simple definition:

Instead of asking large language models to learn everything from their own memory during training, which may give incorrect answers, we use the Retrieval-Augmented Generation system.

---

## Official definition:

It is a two-stage architecture where the Retriever (usually based on Vector DB) retrieves documents relevant to the query from the knowledge base.

And finally uses these documents as context to produce the final answer.

## Why is it important?

- Large language models have outdated information and only have information at training time.
- Using fine-tuning to update the knowledge of large language models requires powerful hardware, is expensive, time-consuming, and makes no sense at all.
- The RAG system solves this problem and if the document changes, only the Vector DB changes.

## What happens without RAG?

The model either gives a false answer (hallucination), or answers honestly: "I don't know," or answers irrelevantly.

### step 1: Loading Document

The first step in building RAG systems is to upload documents.

#### Simple definition:

The first step in any RAG system. That is, we read the raw files (PDF, Word, HTML, scanned image, web page, etc.) from where they are (disk, database, S3, API) and convert them into a uniform format (usually text + metadata) so that the next steps (Parsing, Chunking) can work on them.

#### Technical definition:

The process of transforming heterogeneous data sources into a standard data structure (e.g., Document object with page_content and metadata fields) that the rest of the pipeline can work with uniformly.

### Why is it important?

What problem does it solve?

- Without uniform loading, each file format requires separate code and the pipeline becomes brittle.
- Why is it used in RAG? Because input to real systems (Enterprise) is almost always multi-format: PDF contracts, emails, Confluence pages, Excel files.
- What happens without it? Either some documents are not entered at all (missing data), or they are loaded in a corrupted format (e.g. scanned PDF with only images and no text) and subsequent steps work on poor quality data → the overall quality of RAG goes down.

⚠️ Key point: Retrieval quality is never better than Loading quality. If you mess up here, no cool Reranker or LLM can compensate.

Let's upload a pdf file.

In [1]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\hosse\AppData\Local\Temp\ipykernel_23624\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
loaders = PyPDFLoader(
    file_path="source/pdf1.pdf"
)

In [3]:
data = loaders.load()

incorrect startxref pointer(1)
parsing for Object Streams


In [4]:
data

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T23:23:16+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T23:23:16+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'source/pdf1.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Employee Handbook\nNon-Disclosure Agreement (NDA) Policy\nEmployees must protect confidential information belonging to the company, its clients, and partners.\nThis includes, but is not limited to, product roadmaps, customer data, internal communications,\nproprietary algorithms, financial information, and unreleased features. Confidential information may not\nbe shared with unauthorized individuals inside or outside the organization. These obligations continue\nafter employment ends.\nWorkplace Conduct Policy\nEmployees must maintain a respectful, professional environment free from harassment, di

In [5]:
print(data[0].page_content)

Employee Handbook
Non-Disclosure Agreement (NDA) Policy
Employees must protect confidential information belonging to the company, its clients, and partners.
This includes, but is not limited to, product roadmaps, customer data, internal communications,
proprietary algorithms, financial information, and unreleased features. Confidential information may not
be shared with unauthorized individuals inside or outside the organization. These obligations continue
after employment ends.
Workplace Conduct Policy
Employees must maintain a respectful, professional environment free from harassment, discrimination,
and intimidation. All employees are expected to follow organizational values, collaborate effectively,
and communicate constructively. Disruptive behavior, verbal abuse, or misuse of company systems is
prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 years of service: 10 days
per

###  Step 2: Parsing

After loading the raw file in the previous lesson and getting a block of raw text, we now need to find the hidden structure in that text — what's a heading, what's a paragraph, what's a table, what's a list. Parsing means turning a "wall of formless text" into a structured document.

Technical definition: 

The process of analyzing raw content and extracting meaningful structural elements (headings, paragraphs, tables, lists, code blocks, images) along with their hierarchy, so that this structure can be used in subsequent steps (Chunking).

---

Why is it important?
- What problem does it solve? The raw text we get from Loading is usually "flat" — meaning we don't know where one section ends and another begins, or which sections are tables and should be treated differently.
- Why is it used in RAG? Because the quality of Chunking (the next step) is completely dependent on us knowing the structure of the document. If we know "this is an H2 Heading", we can organize the Chunks more logically (e.g. keep each Chunk under a Heading).

#### What happens without it? 

Tables end up as jumbled text (rows stuck together with odd spacing), titles don't separate from the text below them, and chunks are produced that are incomprehensible to both humans and LLM.

Let's write the code for a parser.

We need to use the `unstructured` library first.

We need to create a structured unit first.

In [6]:
from dataclasses import dataclass, field

In [7]:
@dataclass
class ParsedElement:
    element_type: str
    content: str
    metadata :dict = field(default_factory=dict)

In [8]:
from unstructured.partition.auto import partition
from unstructured.documents.elements import Title, Table, ListItem, NarrativeText

In [9]:
def parse_documnet(file_path: str)->list[ParsedElement]:
    raw_elements = partition(filename=file_path, strategy="fast")
    
    parsed = []
    for el in raw_elements:
        if isinstance(el, Title):
            el_type = "title"
        elif isinstance(el, Table):
            el_type = "table"
        elif isinstance(el, ListItem):
            el_type = "list_item"
        elif isinstance(el, NarrativeText):
            el_type = "narrative_text"
        
        parsed.append(
            ParsedElement(
                element_type=el_type,
                content=str(el),
                metadata={
                    "page_number": el.metadata.page_number,
                    "source_file": file_path
                }
            )
        )
    return parsed

In [10]:
elements = parse_documnet(file_path="source/pdf1.pdf")

incorrect startxref pointer(1)
parsing for Object Streams
No languages specified, defaulting to English.


In [11]:
elements

[ParsedElement(element_type='title', content='Employee Handbook', metadata={'page_number': 1, 'source_file': 'source/pdf1.pdf'}),
 ParsedElement(element_type='title', content='Non-Disclosure Agreement (NDA) Policy', metadata={'page_number': 1, 'source_file': 'source/pdf1.pdf'}),
 ParsedElement(element_type='narrative_text', content='Employees must protect confidential information belonging to the company, its clients, and partners. This includes, but is not limited to, product roadmaps, customer data, internal communications, proprietary algorithms, financial information, and unreleased features. Confidential information may not be shared with unauthorized individuals inside or outside the organization. These obligations continue after employment ends.', metadata={'page_number': 1, 'source_file': 'source/pdf1.pdf'}),
 ParsedElement(element_type='title', content='Workplace Conduct Policy', metadata={'page_number': 1, 'source_file': 'source/pdf1.pdf'}),
 ParsedElement(element_type='narra

In [12]:
for i, el in enumerate(elements):
    print(f"element {i}: {el.element_type}")
    print(f"content:\n{el.content}")
    print(f"metadata: \n{el.metadata}")
    print()

element 0: title
content:
Employee Handbook
metadata: 
{'page_number': 1, 'source_file': 'source/pdf1.pdf'}

element 1: title
content:
Non-Disclosure Agreement (NDA) Policy
metadata: 
{'page_number': 1, 'source_file': 'source/pdf1.pdf'}

element 2: narrative_text
content:
Employees must protect confidential information belonging to the company, its clients, and partners. This includes, but is not limited to, product roadmaps, customer data, internal communications, proprietary algorithms, financial information, and unreleased features. Confidential information may not be shared with unauthorized individuals inside or outside the organization. These obligations continue after employment ends.
metadata: 
{'page_number': 1, 'source_file': 'source/pdf1.pdf'}

element 3: title
content:
Workplace Conduct Policy
metadata: 
{'page_number': 1, 'source_file': 'source/pdf1.pdf'}

element 4: narrative_text
content:
Employees must maintain a respectful, professional environment free from harassment

### Step 3: Cleaning

After parsing, the text has noise:
- Additional page number
- Duplicate header/footer
- additional whitespace
- Strange control characters

***We need to handle these problems because the retrieval quality will decrease if we skip this step.***

---

#### Why do we use RAG?

- It improves the quality of embedding (if half of a chunk contains duplicate page numbers and codes, the embedding vector of that chunk will tend towards noise and the similarity to the real meaning will be less.)
- Lower cost: Each additional character means more tokens and more embedding vectors, and ultimately more resource consumption.

Let's write a cleaner.

We will use regex.

In [13]:
import re
from collections import Counter
from collections import defaultdict

In [14]:
def detect_repeated_boilerplate(
    elements,
    min_repeat_ratio=0.5
):

    page_count = len({
        el.metadata.get("page_number")
        for el in elements
    })

    if page_count <= 1:
        return set()


    text_pages = defaultdict(set)


    for el in elements:
        text = el.content.strip()

        if len(text) < 150 and text:
            page = el.metadata.get("page_number")
            text_pages[text].add(page)


    boilerplate = {
        text
        for text, pages in text_pages.items()
        if len(pages) / page_count >= min_repeat_ratio
    }

    return boilerplate

In [15]:
print(detect_repeated_boilerplate(elements=elements, min_repeat_ratio=0.5))

set()


In [16]:
def clean_text(
    text: str
)->str:
    # Consolidating multiple consecutive intervals into one interval
    text = re.sub(pattern=r'[ \t]+', repl=" ", string=text)
    text = re.sub(pattern=r'\n{3,}', repl='\n\n', string=text)
    return text.strip()

In [17]:
from typing import Callable

In [18]:
def clean_documnet(
    elements:list[ParsedElement]
)-> list[ParsedElement]:
    boilerplate = detect_repeated_boilerplate(elements=elements, min_repeat_ratio=1.5)
    
    cleaned_text = []
    for el in elements:
        striped = el.content.strip()
        
        if striped in boilerplate:
            continue
        
        cleaned_content = clean_text(text=el.content)
        if not cleaned_content:
            continue
        
        cleaned_text.append(
            ParsedElement(
                element_type=el.element_type,
                content=cleaned_content,
                metadata=el.metadata
            )
        )
    return cleaned_text

In [19]:
cleaned_documnets = clean_documnet(
    elements=elements
)

In [20]:
for i, el in enumerate(cleaned_documnets):
    print(f"element {i}: {el.element_type}")
    print(f"content:\n{el.content}")
    print(f"metadata: \n{el.metadata}")
    print()

element 0: title
content:
Employee Handbook
metadata: 
{'page_number': 1, 'source_file': 'source/pdf1.pdf'}

element 1: title
content:
Non-Disclosure Agreement (NDA) Policy
metadata: 
{'page_number': 1, 'source_file': 'source/pdf1.pdf'}

element 2: narrative_text
content:
Employees must protect confidential information belonging to the company, its clients, and partners. This includes, but is not limited to, product roadmaps, customer data, internal communications, proprietary algorithms, financial information, and unreleased features. Confidential information may not be shared with unauthorized individuals inside or outside the organization. These obligations continue after employment ends.
metadata: 
{'page_number': 1, 'source_file': 'source/pdf1.pdf'}

element 3: title
content:
Workplace Conduct Policy
metadata: 
{'page_number': 1, 'source_file': 'source/pdf1.pdf'}

element 4: narrative_text
content:
Employees must maintain a respectful, professional environment free from harassment

### Step 5: Metadata

Simple definition: 

Each chunk of text, in addition to the content itself, has a series of "side tags" that tell you where this piece came from, who it belongs to, what department it belongs to. Like a label on the back of a dress: material, size, production date — the dress itself is one thing, the tag is another, helping you quickly filter through thousands of clothes.

Technical definition: 

A set of structured key-value pairs that are stored with each unit of content (Document or later Chunk) and provide information about the origin, context, and properties of that content — without themselves being part of the text being embedded.

#### Why is it important?

- What problem does it solve? Without metadata, the only way to filter documents is through semantic search within the text itself — which is both slow and not accurate for precise filters `(“only safety department documents,” “only after a certain date”)`.
- Why is it used in RAG?It has two key uses:
    - Metadata Filtering (which we see in Stage 2): Before or at the same time as Vector Search, it limits the search space (for example, search only in documents of a department).
    - Citation / Source Attribution (Stage 4 — Generation): When LLM returns an answer, we need to be able to say "This answer came from file X, page Y" — this is only possible with the right Metadata.
- What happens without it? The system can't tell where the answer came from (reliability decreases), and each search must search the entire knowledge base, which is both more expensive and less accurate when the user wants to search a specific range.

---

⚡**Important note**: 

Metadata is not a "linear and separate" stage like the others — it's a parallel layer that starts at the very beginning of Loading (remember in Lesson 1, we set ```doc.metadata["source_file"]``` ?) and gets richer at each stage (Parsing, Cleaning, Chunking). We've included it here as a separate lesson because it's time to officially talk about its schema design.



### Example:

```python
from pydantic import BaseModel, Field
from datetime import date
from enum import Enum


class DocumentType(str, Enum):
    INCIDENT_REPORT = "incident_report"
    TECHNICAL_SPEC = "technical_spec"
    POLICY = "policy"
    MEETING_NOTES = "meeting_notes"


class ChunkMetadata(BaseModel):
    """
    Formal metadata schema for each Chunk.
    Using Pydantic means automatic validation + type safety
    which prevents silent errors at the scale of 100k documents.
    """
    # --- Provenance ---
    source_file: str = Field(..., description="Original file path or name")
    document_id: str = Field(..., description="Unique document identifier in the system")
    page_number: int | None = Field(None, description="Page number in the original document")

    # --- Organizational Context ---
    department: str | None = Field(None, description="Related Department")
    document_type: DocumentType | None = None

    # --- Time (Temporal) ---
    created_date: date | None = None
    ingested_at: date = Field(default_factory=date.today)

    # --- Structural (inherited from Parsing) ---
    section_title: str | None = Field(None, description="The title of the section this chunk is under.")
    element_type: str = Field(..., description="title | table | narrative_text | list_item")

# Usage example
metadata = ChunkMetadata(
    source_file="incident_report_2024_07.pdf",
    document_id="doc_00042",
    page_number=4,
    department="production_line_3",
    document_type=DocumentType.INCIDENT_REPORT,
    created_date=date(2024, 7, 15),
    section_title="2. Accident statistics",
    element_type="table",
)

print(metadata.model_dump())

```

We don't need any special metadata in this document.

We'll skip it.

### Chunking

Now that we have loaded, parsed, cleaned the document, and designed the metadata, it is time to break the document into smaller pieces (“chunks”). Why? Because neither Embedding Models nor LLM can (or should) process a 50-page document in one go — we need to break it into chunks, each small enough to process accurately, but large enough not to lose meaning.

Technical definition: 

The process of dividing cleaned and structured content into smaller units (chunks), each of which can be embedded and retrieved independently, such that each chunk is sufficiently self-contained (semantically self-sufficient).

#### Why is it important?

- What problem does it solve? 

Embedding models usually have an input length limit (e.g. 8192 tokens), and more importantly, the quality of the embedding degrades when the text is too long (because the final vector has to "average" the meaning of the entire text and details are lost).

- Why is it used in RAG? 

Because Retrieval needs to be able to find exactly the part of the document that is relevant to the user's query — not the entire document, not a single sentence.

- What happens without it? Two extremes of bad things happen:
    - Chunk too large (whole document = 1 Chunk): When a user asks "How many days of PTO?", the entire Employee Handbook (including NDA, Conduct, Travel, which have nothing to do with PTO) is given to LLM as context — lots of noise, high token cost, and the exact answer is likely to get lost among irrelevant information.
    - Chunk too small (each sentence = 1 Chunk): The sentence "Full-time employees accrue PTO according to the following schedule" is separated from the next sentence that actually contains the PTO numbers — and if only one of the two is retrieved, the answer is incomplete.

---

This is the last stage of Stage 1: Document Processing. 

After this, we move on to Stage 2: Retrieval (which starts with Embedding and Vector Database). Chunking is the bridge between “document preparation” and “document searchability.”

### Types of chunking strategies:

- Fixed-Size Chunking
- Recursive Character Splitter
- oken-Based Chunking
- Sentence or Semantic Chunking
- Document-Based Chunking

### Fixed-Size Chunking:

Fixed-size chunking is a data preparation technique that splits large blocks of text into equal-sized segments based on a predefined number of characters, words, or tokens.

#### How It Works؟

- Predefined limits: You set a strict upper bound, such as 500 characters or 256 tokens per chunk.
- Mechanical splitting: The system cuts the text at exact intervals without analyzing grammar, meaning, or sentence boundaries.
- Chunk overlap: Many implementations include a small amount of overlapping text (typically 10% to 20%) between adjacent segments to prevent complete context loss at the boundaries.

#### Advantages
- Speed: It is computationally cheap and processes large volumes of documents instantly because it does not require an AI model to read or interpret the text.
- Predictability: Output sizes are uniform, making database storage calculations, cost estimation, and context-window management straightforward.
- Simplicity: It requires minimal configuration and serves as a reliable baseline for search or Retrieval-Augmented Generation (RAG) pipelines.

#### Disadvantages
- Loss of context: Cutting text blindly can split a sentence, phrase, or critical thought right down the middle, confusing downstream models.
- Lower accuracy: Because semantic boundaries are ignored, important relationships between ideas may get severed across different chunks.

Let's write the code.

In [21]:
from langchain_text_splitters import CharacterTextSplitter

In [22]:
from dataclasses import dataclass, field

In [23]:
@dataclass
class Chunk:
    content: str
    metadata: dict = field(default_factory=dict)

In [24]:
splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=70,
    separator="\n"
)

In [25]:
chunk = [
    chunk 
    for doc in data
    for chunk in splitter.split_text(doc.page_content) 
]

In [26]:
for i ,ch in enumerate(chunk):
    print(f"chunk {i}:")
    print(ch, end="\n\n" )

chunk 0:
Employee Handbook
Non-Disclosure Agreement (NDA) Policy
Employees must protect confidential information belonging to the company, its clients, and partners.
This includes, but is not limited to, product roadmaps, customer data, internal communications,
proprietary algorithms, financial information, and unreleased features. Confidential information may not
be shared with unauthorized individuals inside or outside the organization. These obligations continue
after employment ends.

chunk 1:
after employment ends.
Workplace Conduct Policy
Employees must maintain a respectful, professional environment free from harassment, discrimination,
and intimidation. All employees are expected to follow organizational values, collaborate effectively,
and communicate constructively. Disruptive behavior, verbal abuse, or misuse of company systems is
prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy

chunk 2:
Paid Time Off (PTO) Policy
Full■time employees accr